# Reproduction Notebook — arXiv Preprint

**Paper:** Recall-Optimised Failure Detection in Industrial Telemetry: A Rolling
Degraded-State Counter Versus a 21-Sensor Random Forest on NASA C-MAPSS FD001

**Author:** Puru Pandey  
**Repo:** https://github.com/Puru2001pandey/industrial-telemetry-analytics-research

---

## Prerequisites

1. Download `train_FD001.txt` from NASA C-MAPSS or the LahiruJayasinghe mirror  
2. Place it at `data/train_FD001.txt`  
3. Run all cells in order  

The notebook raises a `FileNotFoundError` if the data file is absent.

In [ ]:
import os, sys
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, recall_score, precision_score, f1_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = 'data/train_FD001.txt'
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Data file not found: {DATA_PATH}\n"
        "Download train_FD001.txt from NASA C-MAPSS and place it in data/"
    )
print(f"Data file found: {DATA_PATH}")

## 1. Load Data

In [ ]:
cols = ['unit_id','cycle','os1','os2','os3'] + [f's{i:02d}' for i in range(1,22)]
df = pd.read_csv(DATA_PATH, sep=r'\s+', header=None, names=cols)

max_cycle = df.groupby('unit_id')['cycle'].max().rename('max_cycle')
df = df.join(max_cycle, on='unit_id')
df['RUL']     = df['max_cycle'] - df['cycle']
df['at_risk'] = (df['RUL'] <= 30).astype(int)

print(f"Dataset: {len(df):,} rows, {df['unit_id'].nunique()} engine units")
print(f"At-risk cycles (RUL<=30): {df['at_risk'].sum():,} ({df['at_risk'].mean()*100:.1f}%)")

## 2. Sensor Preprocessing — Per-Unit Min-Max Normalisation

In [ ]:
# Sensors with meaningful variance in FD001 (Ramasso & Saxena 2014)
RISING  = ['s02','s03','s04','s07','s08','s09','s11','s12','s13','s14','s15']
FALLING = ['s17','s20','s21']
KEEP    = RISING + FALLING

# Per-unit min-max normalisation (removes engine-specific baseline offsets)
for s in KEEP:
    df[s] = df.groupby('unit_id')[s].transform(
        lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
    )

print(f"Normalised {len(KEEP)} sensors: {len(RISING)} rising + {len(FALLING)} falling")

## 3. Health-Window Feature Engineering

In [ ]:
WINDOW_SIZE_CYCLES = 10   # rolling window length
DEGRADED_PCT       = 70   # percentile threshold for 'degraded' label

# Composite degradation score per cycle
df['deg_score'] = (df[RISING].mean(axis=1) + (1 - df[FALLING].mean(axis=1))) / 2.0

# Health-window: count of degraded cycles in rolling window (per engine)
# NOTE: Q_0.70 is computed over the full engine trajectory (post-mortem analysis).
# In live deployment, threshold should be estimated from healthy-phase data only.
df['hw'] = df.groupby('unit_id')['deg_score'].transform(
    lambda x: (x >= np.percentile(x, DEGRADED_PCT)).astype(int)
              .rolling(WINDOW_SIZE_CYCLES, min_periods=1).sum()
)

print(f"Health-window feature: {WINDOW_SIZE_CYCLES}-cycle window, {DEGRADED_PCT}th-percentile threshold")
print(f"Feature range: hw in [0, {int(df['hw'].max())}]")

## 4. Unit-Level 80/20 Train/Test Split

In [ ]:
np.random.seed(42)
units     = df['unit_id'].unique()
np.random.shuffle(units)
train_u   = set(units[:80])
test_u    = set(units[80:])

train = df[df['unit_id'].isin(train_u)]
test  = df[df['unit_id'].isin(test_u)]

print(f"Train: {len(train_u)} engines, {len(train):,} cycles")
print(f"Test:  {len(test_u)} engines, {len(test):,} cycles")
print(f"Test at-risk cycles: {test['at_risk'].sum():,} ({test['at_risk'].mean()*100:.1f}%)")

## 5. Model A — Logistic Regression on 1 Feature

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
lr.fit(train[['hw']], train['at_risk'])
yp_lr = lr.predict(test[['hw']])

print("=== Logistic Regression (health-window, 1 feature) ===")
print(classification_report(test['at_risk'], yp_lr,
                             target_names=['Healthy','At-Risk'], digits=4))

## 6. Model B — Random Forest on 21 Raw Sensors

In [ ]:
rf = RandomForestClassifier(100, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(train[KEEP], train['at_risk'])
yp_rf = rf.predict(test[KEEP])

print("=== Random Forest (21 normalised sensor channels) ===")
print(classification_report(test['at_risk'], yp_rf,
                             target_names=['Healthy','At-Risk'], digits=4))

## 7. Summary Table (Table 1 in paper)

In [ ]:
lr_rec = recall_score(test['at_risk'], yp_lr, pos_label=1)
rf_rec = recall_score(test['at_risk'], yp_rf, pos_label=1)
lr_f1  = f1_score(test['at_risk'], yp_lr, average='weighted')
rf_f1  = f1_score(test['at_risk'], yp_rf, average='weighted')

print(f"{'Model':<40} {'At-Risk Recall':>15} {'Weighted F1':>13}")
print('-'*70)
print(f"{'LR -- health-window (1 feature)':<40} {lr_rec:>15.4f} {lr_f1:>13.4f}")
print(f"{'RF -- 21 normalised sensor channels':<40} {rf_rec:>15.4f} {rf_f1:>13.4f}")
print()
print(f"LR recall advantage: {lr_rec - rf_rec:+.4f}")

## 8. Statistical Stability — 30 Random Seeds

In [ ]:
print("Running 30-seed stability test (may take ~60s)...")
lr_recs, rf_recs = [], []

for seed in range(30):
    rng = np.random.RandomState(seed)
    sh  = rng.permutation(units)
    tr_u, te_u = set(sh[:80]), set(sh[80:])
    tr = df[df['unit_id'].isin(tr_u)]
    te = df[df['unit_id'].isin(te_u)]

    m_lr = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
    m_lr.fit(tr[['hw']], tr['at_risk'])
    lr_recs.append(recall_score(te['at_risk'], m_lr.predict(te[['hw']]), pos_label=1))

    m_rf = RandomForestClassifier(100, class_weight='balanced', random_state=42, n_jobs=-1)
    m_rf.fit(tr[KEEP], tr['at_risk'])
    rf_recs.append(recall_score(te['at_risk'], m_rf.predict(te[KEEP]), pos_label=1))

lr_arr, rf_arr = np.array(lr_recs), np.array(rf_recs)
t, p = stats.ttest_rel(lr_arr, rf_arr)

print(f"LR: {lr_arr.mean():.4f} +/- {lr_arr.std():.4f}")
print(f"RF: {rf_arr.mean():.4f} +/- {rf_arr.std():.4f}")
print(f"Paired t-test: t={t:.4f}, p={p:.6f}")
print(f"LR > RF in {(lr_arr > rf_arr).sum()} / 30 splits")